# Notebook 01 — Data Loading
**Battery-PIAI-ECM v2.0.0**

Loads NASA MAT files for B0005, B0006, B0007, B0018 and extracts discharge cycle records.
Each record contains: Capacity_Ah, Re, Rct (from EIS), voltage/temperature/current statistics.


In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib.pyplot as plt

from src.data_loader import load_all_batteries
from src.config import PROCESSED_CSV, TABLES_DIR


## 1.1 Load all batteries
Place `B0005.mat`, `B0006.mat`, `B0007.mat`, `B0018.mat` in `data/raw/`.


In [ ]:
df = load_all_batteries()
print(f'Total rows: {len(df)}')
df.head(10)


## 1.2 Save processed DataFrame


In [ ]:
import os
os.makedirs(TABLES_DIR, exist_ok=True)
df.to_csv(PROCESSED_CSV, index=False)
print(f'Saved: {PROCESSED_CSV}')


## 1.3 Quick overview plot


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
COLORS = {'B0005':'#2166AC','B0006':'#D6604D','B0007':'#4DAC26','B0018':'#8073AC'}

for bat, g in df.groupby('Battery'):
    g = g.sort_values('CycleIndex')
    axes[0].plot(g['CycleIndex'], g['Capacity_Ah'], color=COLORS[bat], label=bat)
    axes[1].plot(g['CycleIndex'], g['Re']*1000,    color=COLORS[bat], label=bat)
    axes[2].plot(g['CycleIndex'], g['Rct']*1000,   color=COLORS[bat], label=bat)

for ax, title, ylabel in zip(axes,
    ['Capacity Fade','Re Trend','Rct Trend'],
    ['Capacity (Ah)','Re (mΩ)','Rct (mΩ)']):
    ax.set_title(title); ax.set_xlabel('Cycle'); ax.set_ylabel(ylabel); ax.legend(fontsize=9)

plt.tight_layout()
plt.show()
